# 04.05 · Hugging Face Hub & open-weight models

**Chapter:** Transformers & How LLMs Work · **Milestone:** M1

## Goals
- [ ] Find, evaluate and download models from the Hub; read model cards and licenses
- [ ] Use `pipeline` for quick tasks and `AutoTokenizer` / `AutoModelForCausalLM` for control
- [ ] Run a small instruct model locally with its chat template
- [ ] Know the major open-weight families

## Resources
- [Hugging Face LLM Course](https://huggingface.co/learn/llm-course) — free
- [Transformers docs](https://huggingface.co/docs/transformers/) · [Hub docs](https://huggingface.co/docs/hub/) — free
- [Open LLM leaderboard / model search](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending) — free

| Family | By | Notes |
|---|---|---|
| **Llama** | Meta | huge ecosystem; custom license with usage terms |
| **Qwen** | Alibaba | strong across sizes (0.6B→200B+), Apache-2.0 for most; thinking modes |
| **Gemma** | Google | small/medium, good multilingual; Gemma license |
| **Mistral / Mixtral** | Mistral AI | efficient, several Apache-2.0 |
| **DeepSeek** | DeepSeek | large MoE models, strong reasoning; MIT for many |
| **gpt-oss** | OpenAI | open-weight reasoning models, Apache-2.0 |
| **Phi / SmolLM** | Microsoft / HF | tiny models for laptops & edge |

Always check the **license** before shipping a model in a product.

In [ ]:
import time

import torch
from huggingface_hub import model_info
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

## 1. Model metadata without downloading
Model ID = `org/name`. The card (README) tells you training data, intended use, license, evals and the chat template.

In [ ]:
MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
info = model_info(MODEL)
print("license:", getattr(info.card_data, "license", None))
print("downloads (30d):", info.downloads, "| tags:", info.tags[:8])
print("files:", [s.rfilename for s in info.siblings])

`.safetensors` = weights (safe format; avoid pickle `.bin` from untrusted repos). `config.json` = architecture.
`tokenizer*.json` + `chat_template` = how to format input.

## 2. `pipeline`: one line, many tasks
Good for quick experiments with task-specific models (classification, NER, summarisation, embeddings, ASR…).

In [ ]:
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", device=device)
print(classifier(["This course is great!", "The API keeps timing out."]))

## 3. A chat model, step by step
1. Tokenizer turns messages into IDs via the chat template (04.04). 2. `generate` samples tokens. 3. Decode only the new tokens.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to(device)
print(f"{model.num_parameters() / 1e6:.0f}M parameters, dtype {model.dtype}")
print(model.config.num_hidden_layers, "layers |", model.config.hidden_size, "hidden |", model.config.num_attention_heads, "heads")


def chat(messages, max_new_tokens=120, **gen_kwargs):
    input_ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].to(device)
    start = time.perf_counter()
    out = model.generate(input_ids, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id, **gen_kwargs)
    new = out[0, input_ids.shape[1]:]
    print(f"[{len(new)} tokens, {len(new) / (time.perf_counter() - start):.0f} tok/s]")
    return tok.decode(new, skip_special_tokens=True)


print(chat([{"role": "user", "content": "Explain what a GPU is in two sentences."}], do_sample=False))

`add_generation_prompt=True` appends the "assistant turn starts here" marker so the model answers instead of continuing the user's text.

## 4. Base vs instruct
The *base* model (pretrained only) treats your text as a document to continue. The *instruct* model was SFT'd + preference-tuned.

In [ ]:
base_tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
base = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M", torch_dtype=torch.bfloat16).to(device)
ids = base_tok("Explain what a GPU is in two sentences.", return_tensors="pt").to(device)
print("BASE:", base_tok.decode(base.generate(**ids, max_new_tokens=60, do_sample=False, pad_token_id=base_tok.eos_token_id)[0], skip_special_tokens=True))

## 5. Memory math
Weights memory ≈ parameters × bytes per parameter. Add KV cache and activations on top.

In [ ]:
for params_b in [0.135, 1.7, 8, 70]:
    print(f"{params_b:>6}B params: fp32 {params_b * 4:6.1f} GB | bf16 {params_b * 2:6.1f} GB | int4 {params_b * 0.5:6.1f} GB")

A 16 GB Mac comfortably runs ~8B models at 4-bit (via Ollama/MLX, chapter 05.06); `transformers` in bf16 is best for small models
or GPUs.

## Exercises
1. Download `Qwen/Qwen3-0.6B` (or another model under 2B) and chat with it via `chat()`. Compare answers and speed with SmolLM2-135M.
2. Find three models on the Hub for: Spanish sentiment analysis, sentence embeddings, speech recognition. Note each license.
3. Print `model` and identify the attention, MLP, and normalisation layers. Which of 04.03's components do you recognise? Which differ (RMSNorm, SwiGLU, RoPE)?

In [ ]:
# your code here